<a href="https://colab.research.google.com/github/fattah-abdullah/ProyekAkhir_5A_CardiovascularDisease/blob/main/P12C_Varian3_Pemilihan_Fitur_Abdullah_Fattah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Varian 3 — Pemilihan Fitur
## Proyek Akhir Pembelajaran Mesin (INF62325) — Cabang A: Regresi Linear

| | |
|---|---|
| Kelompok | 6 |
| Anggota penanggung jawab | Abdullah Fattah 2488010067 |

---

### Panduan notebook ini

**Notebook ini milik Anda sendiri.** Dua anggota lain mengerjakan varian mereka
di notebook terpisah, boleh bersamaan. Yang Anda hasilkan dibandingkan dengan
milik mereka di Pertemuan 13.

**Masukan.** `p12_data.joblib` dari notebook bersama `P12_Data_Bersama.ipynb`.
Bila berkas itu belum ada, notebook ini berhenti di sel pemeriksaan.

**Varian Anda: fitur numerik disaring menurut korelasi, kolom kategorik ditambahkan**

Kolom numerik disaring berdasarkan korelasinya terhadap target, lalu
kolom kategorik ditambahkan lewat pengodean one-hot.

Anda satu-satunya anggota yang menentukan sebuah angka sendiri:
`AMBANG_KORELASI`. Anda juga satu-satunya yang memakai kolom
kategorik — pada banyak dataset, justru itulah sebab utama hasil
varian ini berbeda jauh dari dua varian lain, bukan penyaringan
fiturnya. Siapkan jawaban untuk itu.

**Luaran.** `varian3.joblib` (model terlatih) dan `varian3_hasil.json`
(angka evaluasinya).

**Satu aturan yang tidak boleh dilanggar.** Jangan menjalankan
`train_test_split` di notebook ini. Pembagian data sudah dilakukan sekali di
notebook bersama, dan notebook ini memeriksanya lewat sidik jari. Varian
1 dan 2 diuji pada data uji yang sama; begitu salah satu
membaginya sendiri, perbandingan di Pertemuan 13 batal.

**Perkiraan waktu.** 60 menit.

---
## Langkah A — Samakan versi scikit-learn (WAJIB, setiap kali membuka notebook ini)

Colab memulai sesi baru setiap kali notebook dibuka, jadi langkah ini diulang
**di setiap pertemuan**, bukan sekali saja.

Gunanya menyamakan versi scikit-learn di Colab dengan versi yang dipasang
Streamlit Community Cloud. Tanpa ini, model berhasil dilatih tetapi gagal dimuat
di aplikasi web dengan galat
`AttributeError: ... has no attribute '_RemainderColsList'`.

In [14]:
!pip install -q scikit-learn==1.9.1

### Restart dulu, jangan dilewati

Pilih menu **Runtime → Restart session**, lalu jalankan ulang dari sel paling atas.
Tanpa restart, Python masih memakai scikit-learn versi lama yang sudah terlanjur
dimuat ke memori, dan kesalahannya tidak terlihat sampai tahap paling akhir.

In [15]:
import sklearn

print("scikit-learn:", sklearn.__version__)
assert sklearn.__version__ == "1.9.1", (
    "Versi belum sesuai. Jalankan sel pip di atas, lalu Runtime -> Restart session, "
    "lalu jalankan ulang dari sel paling atas."
)
print("Versi sudah sesuai.")

scikit-learn: 1.9.1
Versi sudah sesuai.


---
## Langkah B — Hubungkan Google Drive

Setiap pertemuan menyimpan hasilnya ke satu folder Drive yang sama, lalu pertemuan
berikutnya membacanya dari sana. Dengan begitu pekerjaan tidak hilang saat sesi Colab
berakhir, dan tidak perlu mengulang dari awal tiap minggu.

Folder yang dipakai: `/content/drive/MyDrive/ProyekAkhirML_RegresiLinear`

Saat sel di bawah dijalankan, Colab meminta izin mengakses Drive. Pilih akun Google
kelompok, lalu setujui.

In [16]:
import os
from google.colab import drive

drive.mount("/content/drive")

FOLDER = "/content/drive/MyDrive/ProyekAkhirML_RegresiLinear"
os.makedirs(FOLDER, exist_ok=True)

print("Folder proyek:", FOLDER)
print("Isi folder   :", sorted(os.listdir(FOLDER)) or "(masih kosong)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder proyek: /content/drive/MyDrive/ProyekAkhirML_RegresiLinear
Isi folder   : ['data_mentah.csv', 'model_terpilih.joblib', 'p12_data.joblib', 'p13_metrik.json', 'varian1.joblib', 'varian1_hasil.json', 'varian2.joblib', 'varian2_hasil.json', 'varian3.joblib', 'varian3_hasil.json']


---
## Langkah C — Muat data bersama

In [17]:
# Memastikan hasil notebook bersama P12_Data_Bersama benar-benar ada sebelum lanjut.
wajib = ["p12_data.joblib"]
hilang = [b for b in wajib if not os.path.exists(os.path.join(FOLDER, b))]

assert not hilang, (
    f"Berkas berikut belum ada di folder Drive: {hilang}. "
    "Jalankan notebook notebook bersama P12_Data_Bersama sampai selesai lebih dulu."
)
print("Semua berkas masukan tersedia:", wajib)

Semua berkas masukan tersedia: ['p12_data.joblib']


In [18]:
import joblib
import json
import numpy as np
import pandas as pd

p12 = joblib.load(os.path.join(FOLDER, "p12_data.joblib"))

X_latih = p12["X_latih"]; X_uji = p12["X_uji"]
y_latih = p12["y_latih"]; y_uji = p12["y_uji"]
kolom_numerik = p12["kolom_numerik"]
kolom_kategorik = p12["kolom_kategorik"]
TARGET = p12["TARGET"]

print("Data latih:", X_latih.shape, "| Data uji:", X_uji.shape)
print("Numerik  :", kolom_numerik)
print("Kategorik:", kolom_kategorik)

Data latih: (55993, 10) | Data uji: (13999, 10)
Numerik  : ['ap_lo', 'height', 'weight', 'age']
Kategorik: ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']


### Pemeriksaan sidik jari

Sel ini membuktikan bahwa data uji Anda sama persis dengan data uji kedua
anggota lain. Kodenya harus sama dengan yang dicetak notebook bersama.

In [19]:
import hashlib

def sidik_jari(X_uji):
    """Sidik jari pembagian data uji.

    Dipakai ketiga notebook varian untuk membuktikan bahwa mereka benar-benar
    memakai data uji yang sama. Bila ada anggota yang menjalankan
    train_test_split sendiri, sidik jarinya berbeda dan notebook berhenti.
    """
    isi = "|".join(str(i) for i in sorted(X_uji.index.tolist()))
    isi += f"|{X_uji.shape[0]}x{X_uji.shape[1]}"
    return hashlib.sha256(isi.encode()).hexdigest()[:16]


sidik_sekarang = sidik_jari(X_uji)
print("Sidik jari data uji :", sidik_sekarang)
print("Tercatat di P12     :", p12["sidik_jari_uji"])

assert sidik_sekarang == p12["sidik_jari_uji"], (
    "Data uji TIDAK sama dengan yang dibuat di P12 bagian bersama.\n"
    "Jangan menjalankan train_test_split di notebook ini. Muat ulang\n"
    "p12_data.joblib apa adanya, lalu jalankan dari sel paling atas."
)
print("\nCocok. Varian 3 diuji pada data uji yang sama dengan varian lain.")

Sidik jari data uji : 3e147073c4769645
Tercatat di P12     : 3e147073c4769645

Cocok. Varian 3 diuji pada data uji yang sama dengan varian lain.


---
## Langkah D — Merakit Pipeline Varian 3

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Menghitung korelasi

Dihitung **dari data latih saja**, dengan alasan yang sama seperti batas IQR
pada Varian 2: memakai seluruh data berarti informasi data uji ikut menentukan
fitur mana yang dipilih.

In [21]:
korelasi_latih = (X_latih[kolom_numerik]
                  .join(y_latih)
                  .corr(numeric_only=True)[TARGET]
                  .drop(TARGET)
                  .abs()
                  .sort_values(ascending=False))

print("Korelasi mutlak terhadap target:")
print(korelasi_latih.round(4).to_string())

Korelasi mutlak terhadap target:
weight    0.0275
age       0.0184
ap_lo     0.0140
height    0.0055


### Menentukan ambang — KEPUTUSAN ANDA

Lihat tabel di atas, lalu pilih ambang yang memisahkan fitur berinformasi dari
fitur yang hampir tidak berhubungan. Bila ada jurang jelas pada daftar —
misalnya 0,78 lalu 0,30 lalu 0,04 — letakkan ambang di jurang itu.

Jangan membiarkan nilai bawaan 0,10 tanpa alasan. Angka itu ada supaya selnya
bisa dijalankan, bukan karena cocok untuk setiap dataset. Tuliskan di sel temuan
berapa ambang yang Anda pilih dan kolom apa yang akibatnya terbuang.

In [22]:
AMBANG_KORELASI = 0.01      # TENTUKAN SENDIRI setelah melihat tabel di atas

fitur_numerik = korelasi_latih[korelasi_latih >= AMBANG_KORELASI].index.tolist()
fitur_kategorik = kolom_kategorik
DIBUANG_FITUR = [k for k in kolom_numerik if k not in fitur_numerik]

print(f"Ambang           : {AMBANG_KORELASI}")
print(f"Numerik dipakai  : {fitur_numerik}")
print(f"Numerik dibuang  : {DIBUANG_FITUR or '(tidak ada)'}")
print(f"Kategorik dipakai: {fitur_kategorik}")

assert fitur_numerik, (
    "Tidak ada fitur numerik yang lolos ambang. Turunkan AMBANG_KORELASI."
)

Ambang           : 0.01
Numerik dipakai  : ['weight', 'age', 'ap_lo']
Numerik dibuang  : ['height']
Kategorik dipakai: ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']


### Merakit Pipeline

Satu argumen yang **wajib ada**: `handle_unknown="ignore"` pada `OneHotEncoder`.
Tanpa itu, aplikasi web di Pertemuan 14 berhenti dengan galat begitu pengguna
memasukkan kategori yang tidak pernah muncul di data latih. Varian 1 dan 2 tidak
terkena hal ini karena tidak memakai kolom kategorik — kalau varian Anda yang
terpilih, kesalahan ini terbawa sampai ke aplikasi.

In [23]:
pipeline = Pipeline([
    ("pre", ColumnTransformer([
        ("num", Pipeline([
            ("imputasi", SimpleImputer(strategy="median")),
            ("skala", StandardScaler()),
        ]), fitur_numerik),
        ("kat", Pipeline([
            ("imputasi", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), fitur_kategorik),
    ])),
    ("reg", LinearRegression()),
])

jumlah_kolom = pipeline.named_steps["pre"].fit_transform(X_latih).shape[1]
print(f"{X_latih.shape[1]} kolom asli -> {jumlah_kolom} kolom setelah preprocessing")
print("Pertambahannya berasal dari one-hot encoding kolom kategorik.")
print("Bila pertambahannya sangat besar (misalnya jadi 900 kolom), ada kolom")
print("berkardinalitas tinggi yang lolos. Kembali ke eksplorasi bersama.")
pipeline

10 kolom asli -> 17 kolom setelah preprocessing
Pertambahannya berasal dari one-hot encoding kolom kategorik.
Bila pertambahannya sangat besar (misalnya jadi 900 kolom), ada kolom
berkardinalitas tinggi yang lolos. Kembali ke eksplorasi bersama.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('pre', ...), ('reg', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('kat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of 

---
## Langkah E — Melatih dan mengukur

Diukur pada `X_uji`, data uji bersama. Inilah angka yang masuk ke tabel
perbandingan Pertemuan 13.

In [24]:
pipeline.fit(X_latih, y_latih)

duga_uji = pipeline.predict(X_uji)
mae = mean_absolute_error(y_uji, duga_uji)
rmse = float(np.sqrt(mean_squared_error(y_uji, duga_uji)))
r2 = r2_score(y_uji, duga_uji)
r2_latih = r2_score(y_latih, pipeline.predict(X_latih))

print(f"Hasil Varian 3 pada DATA UJI")
print(f"  MAE  = {mae:,.2f}")
print(f"  RMSE = {rmse:,.2f}")
print(f"  R2   = {r2:.4f}")
print(f"\nR2 pada data latih = {r2_latih:.4f}  "
      f"(selisih {r2_latih - r2:+.4f})")
print("Selisih besar menandakan model menghafal data latihnya.")

Hasil Varian 3 pada DATA UJI
  MAE  = 12.86
  RMSE = 25.04
  R2   = 0.0576

R2 pada data latih = 0.0015  (selisih -0.0560)
Selisih besar menandakan model menghafal data latihnya.


### Validasi silang

In [25]:
skor = cross_val_score(pipeline, X_latih, y_latih, cv=5, scoring="r2")

cv_rata = float(skor.mean())
cv_simpangan = float(skor.std())
print(f"Validasi silang 5 lipatan Varian 3")
print(f"  rata-rata R2   = {cv_rata:.4f}")
print(f"  simpangan baku = {cv_simpangan:.4f}")
print(f"  tiap lipatan   = {skor.round(3)}")
print("\nSimpangan baku besar berarti angka tunggal pada data uji")
print("tidak boleh dipercaya sepenuhnya. Catat angka ini.")

Validasi silang 5 lipatan Varian 3
  rata-rata R2   = 0.0014
  simpangan baku = 0.0009
  tiap lipatan   = [0.002 0.002 0.    0.002 0.   ]

Simpangan baku besar berarti angka tunggal pada data uji
tidak boleh dipercaya sepenuhnya. Catat angka ini.


### Temuan Anda — ISI SEL INI

Ini yang Anda pertanggungjawabkan sendiri saat tanya jawab. Sebutkan angkanya.

- Fitur yang varian saya pakai: untuk fitur numerik saya memakai: `weight`, `age` dan `ap_lo`. Lalu untuk fitur kategorik saya memakai: `gender`, `cholesterol`, `gluc`, `smoke`, `alco`, dan `active`.
- Yang saya perlakukan berbeda dari varian lain: saya melakukan seleksi pada fitur numerik (menghapus `height`) berdasarkan nilai korelasi absolut, yaitu minimal **0,01** terhadap target ap_hi. setelah itu saya juga melakukan One-Hot Encoding pada fitur kategorikal agar setiap kategori dapat direpresentasikan sebagai variabel dummy yang dapat digunakan oleh model Regresi Linear.
- Hasil pada data uji — MAE, RMSE, R²: MAE sebesar **12.86**, RMSE sebesar **25.04**, dan R² sebesar **0.0576**
- Kestabilan menurut validasi silang: Berdasarkan 5-Fold Cross Validation, diperoleh rata-rata R² sebesar **0,0014** dengan simpangan baku sebesar **0,0009**.
- Dugaan saya mengapa hasilnya begini:<br>Rendahnya performa model diduga karena hubungan linear antara sebagian besar fitur dengan target tekanan darah sistolik masih lemah. Selain itu, adanya nilai ekstrem pada target ap_hi kemungkinan memengaruhi kesalahan prediksi, terutama pada RMSE. Penggunaan One-Hot Encoding pada fitur kategorikal juga belum tentu memberikan kontribusi prediksi yang signifikan.

Dugaan boleh salah. Pertemuan 13 yang akan membuktikannya, dan yang dinilai
adalah apakah Anda paham **apa** yang Anda ubah dan **mengapa** perubahan itu
bisa berpengaruh.

---
## Langkah F — Menyimpan hasil ke Drive

In [26]:
ANGGOTA = "(yeye, 1234)"      # ISI dengan nama dan NIM Anda

hasil = {
    "varian": 3,
    "nama": "Pemilihan Fitur",
    "anggota": ANGGOTA,
    "fitur_numerik": fitur_numerik,
    "fitur_kategorik": fitur_kategorik,
    "baris_latih": int(len(X_latih)),
    "mae": float(mae),
    "rmse": float(rmse),
    "r2": float(r2),
    "r2_latih": float(r2_latih),
    "cv_rata": cv_rata,
    "cv_simpangan": cv_simpangan,
    "sidik_jari_uji": sidik_sekarang,
    "ambang_korelasi": AMBANG_KORELASI,
    "numerik_dibuang": DIBUANG_FITUR,
    "catatan": f"Fitur numerik disaring pada ambang korelasi {AMBANG_KORELASI}, kolom kategorik ditambahkan lewat one-hot.",
}

assert ANGGOTA != "(nama, NIM)", (
    "Isi ANGGOTA dengan nama dan NIM Anda di baris paling atas sel ini,\n"
    "lalu jalankan ulang. Berkas sengaja belum ditulis: tanpa nama,\n"
    "Pertemuan 13 tidak bisa menyebut siapa pemilik varian ini."
)

joblib.dump(pipeline, os.path.join(FOLDER, "varian3.joblib"))
with open(os.path.join(FOLDER, "varian3_hasil.json"), "w", encoding="utf-8") as f:
    json.dump(hasil, f, indent=2, ensure_ascii=False)

print("Tersimpan: varian3.joblib dan varian3_hasil.json")
print(json.dumps(hasil, indent=2, ensure_ascii=False))
print("\nIsi folder:", sorted(os.listdir(FOLDER)))

Tersimpan: varian3.joblib dan varian3_hasil.json
{
  "varian": 3,
  "nama": "Pemilihan Fitur",
  "anggota": "(yeye, 1234)",
  "fitur_numerik": [
    "weight",
    "age",
    "ap_lo"
  ],
  "fitur_kategorik": [
    "gender",
    "cholesterol",
    "gluc",
    "smoke",
    "alco",
    "active"
  ],
  "baris_latih": 55993,
  "mae": 12.864796258369688,
  "rmse": 25.043192511048503,
  "r2": 0.05755837544136633,
  "r2_latih": 0.0015220660653546547,
  "cv_rata": 0.001382356418780839,
  "cv_simpangan": 0.0008748450889870227,
  "sidik_jari_uji": "3e147073c4769645",
  "ambang_korelasi": 0.01,
  "numerik_dibuang": [
    "height"
  ],
  "catatan": "Fitur numerik disaring pada ambang korelasi 0.01, kolom kategorik ditambahkan lewat one-hot."
}

Isi folder: ['data_mentah.csv', 'model_terpilih.joblib', 'p12_data.joblib', 'p13_metrik.json', 'varian1.joblib', 'varian1_hasil.json', 'varian2.joblib', 'varian2_hasil.json', 'varian3.joblib', 'varian3_hasil.json']


---
## Isi folder Drive setelah notebook ini

```
ProyekML_RegresiLinear/          ← folder Google Drive kelompok
├── data_mentah.csv              · dari bagian bersama
├── p12_data.joblib              · dari bagian bersama, DIBACA HARI INI
├── varian3.joblib              ← DIHASILKAN NOTEBOOK INI
└── varian3_hasil.json          ← DIHASILKAN NOTEBOOK INI
```

Dua berkas milik anggota lain (`varian1.*` dan `varian2.*`) muncul di folder
yang sama ketika mereka selesai. Ketiganya tidak saling menimpa.

## Daftar periksa

- [X] Nama dan NIM terisi di sel identitas dan di `ANGGOTA`
- [X] Sel sidik jari mencetak `Cocok`
- [X] `train_test_split` **tidak** dijalankan di notebook ini
- [X] `AMBANG_KORELASI` ditentukan sendiri, beserta alasannya
- [X] `handle_unknown="ignore"` ada pada `OneHotEncoder`
- [X] Jumlah kolom setelah preprocessing sudah diperiksa dan masuk akal
- [X] Sel temuan terisi dengan angka
- [X] Sel Langkah F mencetak `Tersimpan: varian3.joblib dan varian3_hasil.json`
- [X] Notebook sudah tersimpan: **File → Save**

Pertemuan 13 baru bisa dijalankan setelah ketiga anggota menyelesaikan
notebooknya masing-masing.